In [1]:
from ortools.algorithms.python import knapsack_solver
import numpy as np
import time
import threading

In [2]:
NUMBER_OF_TRIALS = 100
PREVIEW_ITEMS = 15

import numpy as np

import problems

# Every implementation in this project solves the identical set of problems by
# reading them from problems.py.
problem_set = problems.generate()

num_problems = problem_set.num_problems
num_items_per_problem = problem_set.num_items
max_capacity = problem_set.max_capacity
capacities = problem_set.capacities

items = problem_set.items

print(f"{num_problems} problems, {num_items_per_problem} items each, capacity at most {max_capacity}")
for i in range(3):
    row = problem_set.items[i]
    head = ", ".join(str(item) for item in row[:PREVIEW_ITEMS])
    rest = f", ... ({len(row) - PREVIEW_ITEMS} more)" if len(row) > PREVIEW_ITEMS else ""
    print(f"Problem {i + 1}: capacity {problem_set.capacities[i]}, items [{head}{rest}]")

10000 problems, 100 items each, capacity at most 100
Problem 1: capacity 82, items [5, 38, 33, 22, 22, 43, 5, 35, 10, 5, 26, 48, 37, 38, 36, ... (85 more)]
Problem 2: capacity 37, items [41, 10, 40, 1, 40, 39, 39, 33, 24, 35, 14, 39, 28, 23, 25, ... (85 more)]
Problem 3: capacity 75, items [18, 45, 25, 35, 23, 14, 38, 48, 13, 39, 13, 36, 39, 23, 37, ... (85 more)]


In [3]:
# OR-Tools Section

# This section should run in Colab T4
import platform
print(platform.node())

# Function to solve a single subset sum problem using OR-Tools
def solve_subset_sum(items, capacity, problem_idx, results):
    solver = knapsack_solver.KnapsackSolver(
        knapsack_solver.KNAPSACK_MULTIDIMENSION_BRANCH_AND_BOUND_SOLVER, 'SubsetSumExample')

    # Subset sum is knapsack with an item's value equal to its weight, so
    # OR-Tools is given the same array for both.
    solver.init(items, [items], [capacity])

    max_value = solver.solve()
    results[problem_idx] = max_value
    #print(f"Problem {problem_idx + 1}: Maximum value = {max_value}")

# Storage for results
results = [0] * num_problems

execution_times = []
for trial in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    # Create and start threads
    threads = []
    for i in range(num_problems):
        t = threading.Thread(target=solve_subset_sum, args=(items[i], capacities[i], i, results))
        threads.append(t)
        t.start()

    # Wait for all threads to complete
    for t in threads:
        t.join()

    # Print execution time
    end_time = time.time()
    #print(f"Threaded execution time: {end_time - start_time:.6f} seconds")
    #print(f"{(trial + 1) * 100 // NUMBER_OF_TRIALS}% complete")
    execution_times.append(end_time - start_time)

print(f"Average CPU execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Print the final results
#for i in range(num_problems):
for i in range(10):
    print(f"Final Result for Problem {i + 1}: Maximum value = {results[i]}")

Andrews-MacBook-Air.local
Average CPU execution time: 0.459661 seconds
Final Result for Problem 1: Maximum value = 82
Final Result for Problem 2: Maximum value = 37
Final Result for Problem 3: Maximum value = 75
Final Result for Problem 4: Maximum value = 6
Final Result for Problem 5: Maximum value = 65
Final Result for Problem 6: Maximum value = 98
Final Result for Problem 7: Maximum value = 38
Final Result for Problem 8: Maximum value = 58
Final Result for Problem 9: Maximum value = 74
Final Result for Problem 10: Maximum value = 78


In [4]:
# Metal Section

# This section should run on the M4 MacBook Air
import Metal

# Metal kernel to solve multiple subset sum problems. One threadgroup solves one
# problem; its threads split the capacities 0..max_capacity between them.
#
# reachable[w] is 1 when some subset of the items seen so far sums to exactly w.
# Adding an item makes w reachable if w - item already was. Each item reads one
# table and writes the other, so no thread can see a value that already includes
# the current item, which would let that item be used twice.
kernel_code = """
#include <metal_stdlib>
using namespace metal;

kernel void subset_sum(device const int *items [[buffer(0)]],
                       device const int *capacities [[buffer(1)]],
                       device int *max_values [[buffer(2)]],
                       constant int &num_items [[buffer(3)]],
                       constant int &max_capacity [[buffer(4)]],
                       threadgroup uchar *tables [[threadgroup(0)]],
                       uint problem_idx [[threadgroup_position_in_grid]],
                       uint thread_idx [[thread_position_in_threadgroup]],
                       uint threads_per_group [[threads_per_threadgroup]]) {
    int width = max_capacity + 1;
    threadgroup uchar *current = tables;
    threadgroup uchar *next = tables + width;

    // Only the empty subset exists before any item is considered
    for (int w = thread_idx; w < width; w += threads_per_group) {
        current[w] = (w == 0);
    }
    threadgroup_barrier(mem_flags::mem_threadgroup);

    device const int *problem_items = items + problem_idx * num_items;
    for (int i = 0; i < num_items; i++) {
        int item = problem_items[i];
        for (int w = thread_idx; w < width; w += threads_per_group) {
            next[w] = current[w] | (w >= item ? current[w - item] : 0);
        }
        threadgroup_barrier(mem_flags::mem_threadgroup);

        threadgroup uchar *swap = current;
        current = next;
        next = swap;
    }

    // The answer is the largest reachable sum that fits in the capacity
    if (thread_idx == 0) {
        int w = capacities[problem_idx];
        while (!current[w]) {
            w--;
        }
        max_values[problem_idx] = w;
    }
}
"""

device = Metal.MTLCreateSystemDefaultDevice()
print(device.name())

# Compile the kernel code
library, error = device.newLibraryWithSource_options_error_(kernel_code, None, None)
if library is None:
    raise RuntimeError(f"Metal kernel failed to compile: {error}")
pipeline, error = device.newComputePipelineStateWithFunction_error_(
    library.newFunctionWithName_("subset_sum"), None)
if pipeline is None:
    raise RuntimeError(f"Metal pipeline could not be created: {error}")
command_queue = device.newCommandQueue()

# Two tables of max_capacity + 1 bytes each; Metal requires a multiple of 16.
table_bytes = 2 * (max_capacity + 1)
threadgroup_memory_size = (table_bytes + 15) // 16 * 16
threads_per_threadgroup = min(max_capacity + 1, pipeline.maxTotalThreadsPerThreadgroup())

execution_times = []
gpu_times = []
for _ in range(NUMBER_OF_TRIALS):
    # Measure execution time
    start_time = time.time()

    # Copy data into GPU buffers. Apple silicon shares memory between the CPU
    # and GPU, but this still copies the arrays, like cuda.memcpy_htod does.
    items_gpu = device.newBufferWithBytes_length_options_(
        items.tobytes(), items.nbytes, Metal.MTLResourceStorageModeShared)
    capacities_gpu = device.newBufferWithBytes_length_options_(
        capacities.tobytes(), capacities.nbytes, Metal.MTLResourceStorageModeShared)
    max_values_gpu = device.newBufferWithLength_options_(
        num_problems * 4, Metal.MTLResourceStorageModeShared)

    # Launch the kernel
    command_buffer = command_queue.commandBuffer()
    encoder = command_buffer.computeCommandEncoder()
    encoder.setComputePipelineState_(pipeline)
    encoder.setBuffer_offset_atIndex_(items_gpu, 0, 0)
    encoder.setBuffer_offset_atIndex_(capacities_gpu, 0, 1)
    encoder.setBuffer_offset_atIndex_(max_values_gpu, 0, 2)
    encoder.setBytes_length_atIndex_(np.int32(num_items_per_problem).tobytes(), 4, 3)
    encoder.setBytes_length_atIndex_(np.int32(max_capacity).tobytes(), 4, 4)
    encoder.setThreadgroupMemoryLength_atIndex_(threadgroup_memory_size, 0)
    encoder.dispatchThreadgroups_threadsPerThreadgroup_(
        Metal.MTLSizeMake(num_problems, 1, 1), Metal.MTLSizeMake(threads_per_threadgroup, 1, 1))
    encoder.endEncoding()
    command_buffer.commit()
    command_buffer.waitUntilCompleted()
    if command_buffer.error() is not None:
        raise RuntimeError(f"Metal kernel failed: {command_buffer.error()}")

    # Copy the result back to the CPU
    max_values = np.frombuffer(
        max_values_gpu.contents().as_buffer(num_problems * 4), dtype=np.int32).copy()

    # Print execution time
    end_time = time.time()
    execution_times.append(end_time - start_time)
    gpu_times.append(command_buffer.GPUEndTime() - command_buffer.GPUStartTime())

print(f"Average Metal execution time: {(sum(execution_times) / NUMBER_OF_TRIALS):.6f} seconds")
print(f"Average Metal kernel time (GPU only): {(sum(gpu_times) / NUMBER_OF_TRIALS):.6f} seconds")

# Check against the OR-Tools section above
mismatches = np.flatnonzero(max_values != np.array(results))
print(f"{len(mismatches)} of {num_problems} results differ from OR-Tools")

# Print the results
#for i in range(num_problems):
for i in range(10):
    print(f"Problem {i + 1}: Maximum value = {max_values[i]}")

Apple M4
Average Metal execution time: 0.003586 seconds
Average Metal kernel time (GPU only): 0.002550 seconds
0 of 10000 results differ from OR-Tools
Problem 1: Maximum value = 82
Problem 2: Maximum value = 37
Problem 3: Maximum value = 75
Problem 4: Maximum value = 6
Problem 5: Maximum value = 65
Problem 6: Maximum value = 98
Problem 7: Maximum value = 38
Problem 8: Maximum value = 58
Problem 9: Maximum value = 74
Problem 10: Maximum value = 78
